# 第 1 週 實作｜極限的嚴格定義與浮點數現實

銜接課教「怎麼算」,這一週回答「越來越靠近到底是什麼意思」——並且揭穿一件不安的事:電腦其實算不出真正的極限。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜數值探極限:先變準,再變爛

銜接課用數值表猜極限。這次把 $h$ 一路縮到 $10^{-16}$,看那張表在什麼時候開始騙人。


In [ ]:
# f(x) = x^2 在 x = 1 的導數,真值 f'(1) = 2
f = lambda x: x**2
x0, exact = 1.0, 2.0

print(f"{'h':>10}  {'差商':>20}  {'誤差':>12}")
hs = [10.0**(-k) for k in range(1, 17)]
errs = []
for h in hs:
    q = (f(x0 + h) - f(x0)) / h
    err = abs(q - exact)
    errs.append(err)
    print(f"{h:10.0e}  {q:20.16f}  {err:12.3e}")

plt.loglog(hs, errs, 'o-')
plt.axvline(np.sqrt(np.finfo(float).eps), color='C3', ls='--',
            label='sqrt(machine eps)')
plt.gca().invert_xaxis()
plt.xlabel('h'); plt.ylabel('|error|'); plt.legend()
plt.title('Forward difference error: down, then up')
plt.show()

best = hs[int(np.argmin(errs))]
print("\n最準的 h =", f"{best:.0e}", "  sqrt(eps) =", f"{np.sqrt(np.finfo(float).eps):.2e}")

In [ ]:
# TODO 學生練習:把 f 換成 sin,x0 換成 0(真值 f'(0) = cos(0) = 1)
# 重跑一次,最佳的 h 還是落在 1e-8 附近嗎?
# f = lambda x: math.sin(x); x0, exact = 0.0, 1.0

## Lab 2｜機器 epsilon:電腦的「最小可分辨距離」

為什麼 $h$ 不能無限小?因為浮點數之間有間距。這格把那個間距量出來。


In [ ]:
eps = np.finfo(float).eps
print("machine epsilon =", eps)
print("eps / 2         =", eps / 2)

# 1 + h 什麼時候才不等於 1?
for k in [15, 16, 17]:
    h = 10.0**(-k)
    print(f"1 + 1e-{k} == 1 ?  {1 + h == 1}   (1+h)-1 = {(1 + h) - 1:.3e}")

# 浮點數的間距隨數字大小改變
for x in [1e-8, 1.0, 1e8, 1e16]:
    print(f"x = {x:8.0e}   相鄰浮點數間距 = {np.spacing(x):.3e}")

# 經典陷阱
print("\n0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)
print("0.1 + 0.2 =", f"{0.1 + 0.2:.20f}")

## Lab 3｜災難性消去:同一個式子,兩種寫法

觀念 10 的數值版本。兩個代數上完全相同的式子,在浮點數下的表現差了好幾個數量級。


In [ ]:
def direct(x):
    return (np.sqrt(1 + x) - 1) / x          # 含相近數相減 → 危險

def stable(x):
    return 1 / (np.sqrt(1 + x) + 1)          # 有理化後只剩加法 → 安全

xs = np.array([10.0**(-k) for k in range(1, 17)])
print(f"{'x':>8}  {'直接算':>20}  {'有理化':>20}")
for x in xs:
    print(f"{x:8.0e}  {direct(x):20.16f}  {stable(x):20.16f}")

err_d = np.abs(direct(xs) - stable(xs))
plt.loglog(xs, np.maximum(err_d, 1e-18), 'o-', label='|direct - stable|')
plt.gca().invert_xaxis()
plt.xlabel('x'); plt.ylabel('discrepancy'); plt.legend()
plt.title('Catastrophic cancellation in (sqrt(1+x)-1)/x')
plt.show()

print("\nx = 1e-16 時 直接算 =", direct(1e-16), " 有理化 =", stable(1e-16))

In [ ]:
# TODO 學生練習:對 sqrt(x+1) - sqrt(x) 做同樣的比較(x 取 1e8, 1e12, 1e16)
# 穩定形式是 1 / (sqrt(x+1) + sqrt(x))
# 兩者在 x = 1e16 差多少?

## Lab 4｜ε-δ 視覺化:把定義畫出來

定義講完之後,用圖把「你給 ε、我找 δ」這件事演一遍。程式會自己找出可行的 δ 並畫框。


In [ ]:
# f(x) = 3x - 1, a = 2, L = 5。理論上 delta = eps / 3
f = lambda x: 3 * x - 1
a, L = 2.0, 5.0

def find_delta(eps, hi=2.0, tol=1e-12):
    """二分搜尋:最大的 delta 使 |x-a|<delta 都滿足 |f(x)-L|<eps"""
    lo = 0.0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        xs = np.linspace(a - mid, a + mid, 400)
        if np.all(np.abs(f(xs) - L) < eps):
            lo = mid
        else:
            hi = mid
    return lo

for eps in [1.0, 0.5, 0.1, 0.01]:
    d = find_delta(eps)
    print(f"eps = {eps:5.2f}  ->  delta ~ {d:.6f}   (理論值 eps/3 = {eps/3:.6f})")

eps = 0.5
d = find_delta(eps)
xs = np.linspace(a - 1, a + 1, 400)
plt.plot(xs, f(xs), lw=2)
plt.axhspan(L - eps, L + eps, alpha=0.2, color='C2', label='y band: 2*eps')
plt.axvspan(a - d, a + d, alpha=0.2, color='C1', label='x window: 2*delta')
plt.plot([a], [L], 'ko')
plt.xlabel('x'); plt.ylabel('f(x)'); plt.legend()
plt.title('Give me eps, I return delta')
plt.show()

In [ ]:
# TODO 學生練習:把 f 換成 x**2、a 換成 1、L 換成 1
# 對 eps = 0.1,程式找到的 delta 是多少?
# 和你用 min{1, eps/3} 手算的保守估計比,哪個大?為什麼手算的比較小?